# Module 06-3: JSON 파서 폴백 전략

## 학습 목표
- LLM 응답에서 JSON을 추출하는 3단계 폴백 전략을 이해할 수 있다
- `parse_json_response()` 함수를 구현할 수 있다
- 3가지 다른 형태의 LLM 응답을 올바르게 파싱할 수 있다

**참조 문서**: `docs/part3-prompt-and-llm/06-structured-output.md` 섹션 2.3

## 개념 요약

### LLM 응답의 3가지 형태

**응답 A** (직접 JSON): `{"status": "PASS", "score": 8}`

**응답 B** (마크다운 코드블록):
```
분석 결과입니다:
```json
{"status": "FAIL", "score": 3}
```
```

**응답 C** (텍스트 속 JSON): `결과를 알려드립니다. {"status": "BLOCKED", "score": 5} 추가 분석 필요`

### 3단계 폴백 전략

```
1단계: json.loads(text.strip())          -- 직접 파싱
2단계: 마크다운 코드블록 regex 추출      -- ```json...``` 패턴
3단계: text.find("{") ~ text.rfind("}")  -- 중괄호 탐색
```

In [ ]:
# 환경 설정
import sys
sys.path.insert(0, '..')

import json
import re

print("환경 설정 완료")

## 실습 1: 3가지 LLM 응답 형태 확인

각 응답 형태를 확인하고 직접 파싱을 시도합니다.

In [ ]:
# 3가지 테스트 입력
text1 = '{"status": "PASS", "score": 8}'  # 직접 JSON
text2 = '분석 결과입니다:\n```json\n{"status": "FAIL", "score": 3}\n```\n추가 분석 중...'  # 마크다운 코드블록
text3 = '결과를 알려드립니다. {"status": "BLOCKED", "score": 5} 추가 분석이 필요합니다.'  # 텍스트 속 JSON

print("=== 테스트 입력 ===")
print(f"text1: {text1}")
print(f"\ntext2: {text2}")
print(f"\ntext3: {text3}")

print("\n=== 직접 파싱 시도 ===")
for i, text in enumerate([text1, text2, text3], 1):
    try:
        result = json.loads(text.strip())
        print(f"text{i}: 성공! {result}")
    except json.JSONDecodeError:
        print(f"text{i}: 실패 (JSONDecodeError)")

In [ ]:
# 검증 셀
try:
    json.loads(text1)
    t1_valid = True
except:
    t1_valid = False
    
try:
    json.loads(text2)
    t2_valid = True
except:
    t2_valid = False

assert t1_valid, "text1은 직접 파싱 가능해야 합니다."
assert not t2_valid, "text2는 직접 파싱 불가여야 합니다."
print("실습 1 완료!")

## 실습 2 (TODO): parse_json_response() 함수 구현

3단계 폴백 전략을 사용하는 `parse_json_response()` 함수를 구현하세요.

In [ ]:
# TODO: 여기에 코드를 작성하세요
#
# 힌트 1 (방향): try/except로 각 단계를 시도합니다.
#               1단계 실패 -> 2단계 시도 -> 2단계 실패 -> 3단계 시도 -> 모두 실패 -> ValueError
#
# 힌트 2 (키워드): json.loads(), re.findall(r"```(?:json)?\s*\n?(.*?)\n?\s*```", text, re.DOTALL),
#               text.find("{"), text.rfind("}")
#
# 힌트 3 (거의 정답):
#   def parse_json_response(text: str) -> dict:
#       # 1단계: 직접 파싱
#       try:
#           return json.loads(text.strip())
#       except json.JSONDecodeError:
#           pass
#
#       # 2단계: 마크다운 코드블록 추출
#       pattern = r"```(?:json)?\s*\n?(.*?)\n?\s*```"
#       matches = re.findall(pattern, text, re.DOTALL)
#       for match in matches:
#           try:
#               return json.loads(match.strip())
#           except json.JSONDecodeError:
#               continue
#
#       # 3단계: 중괄호 블록 탐색
#       start = text.find("{")
#       end = text.rfind("}")
#       if start != -1 and end != -1 and end > start:
#           try:
#               return json.loads(text[start:end + 1])
#           except json.JSONDecodeError:
#               pass
#
#       raise ValueError(f"JSON 파싱 실패. 원본: {text[:200]}...")

def parse_json_response(text: str) -> dict:
    """LLM 응답에서 JSON을 추출한다.
    
    3단계 폴백 전략:
    1) 직접 JSON 파싱 시도
    2) 마크다운 코드블록(```json ... ```) 내부 추출
    3) 첫 번째 { ... } 블록 탐색
    
    Args:
        text: LLM의 원본 응답 텍스트
    
    Returns:
        파싱된 딕셔너리
    
    Raises:
        ValueError: 모든 전략 실패 시
    """
    # TODO: 3단계 폴백 전략 구현
    pass

In [ ]:
# 2-2. 3가지 테스트 입력으로 검증
print("=== parse_json_response() 테스트 ===")

test_cases = [
    ("text1 (직접 JSON)", text1),
    ("text2 (마크다운 코드블록)", text2),
    ("text3 (텍스트 속 JSON)", text3),
]

for name, text in test_cases:
    try:
        result = parse_json_response(text)
        print(f"{name}: 성공!")
        print(f"  결과: {result}")
    except Exception as e:
        print(f"{name}: 실패 - {e}")

In [ ]:
# 검증 셀
assert parse_json_response is not None, "parse_json_response 함수를 구현하세요."

# text1: 직접 JSON 파싱 (1단계)
result1 = parse_json_response(text1)
assert result1 == {"status": "PASS", "score": 8}, f"text1 파싱 결과가 올바르지 않습니다. 현재: {result1}"

# text2: 마크다운 코드블록 추출 (2단계)
result2 = parse_json_response(text2)
assert result2 == {"status": "FAIL", "score": 3}, f"text2 파싱 결과가 올바르지 않습니다. 현재: {result2}"

# text3: 중괄호 탐색 (3단계)
result3 = parse_json_response(text3)
assert result3 == {"status": "BLOCKED", "score": 5}, f"text3 파싱 결과가 올바르지 않습니다. 현재: {result3}"

# 파싱 불가능한 경우 ValueError 발생 확인
try:
    parse_json_response("이것은 JSON이 전혀 없는 텍스트입니다.")
    assert False, "ValueError가 발생해야 합니다."
except ValueError:
    pass

print("실습 2 완료!")

## 실습 3: 어느 단계에서 성공했는지 추적

각 입력이 몇 번째 단계에서 파싱되었는지 추적하는 버전을 만들어봅니다.

In [ ]:
# 3-1. 단계 추적이 포함된 버전
def parse_json_response_with_stage(text: str) -> tuple[dict, int]:
    """JSON을 파싱하고 성공한 단계를 함께 반환한다.
    
    Returns:
        (파싱된 딕셔너리, 성공한 단계 번호)
    """
    # 1단계: 직접 파싱
    try:
        return json.loads(text.strip()), 1
    except json.JSONDecodeError:
        pass

    # 2단계: 마크다운 코드블록 추출
    pattern = r"```(?:json)?\s*\n?(.*?)\n?\s*```"
    matches = re.findall(pattern, text, re.DOTALL)
    for match in matches:
        try:
            return json.loads(match.strip()), 2
        except json.JSONDecodeError:
            continue

    # 3단계: 중괄호 블록 탐색
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(text[start:end + 1]), 3
        except json.JSONDecodeError:
            pass

    raise ValueError(f"JSON 파싱 실패")

# 3-2. 테스트
print("=== 단계 추적 테스트 ===")
for name, text in [("text1", text1), ("text2", text2), ("text3", text3)]:
    result, stage = parse_json_response_with_stage(text)
    stage_names = {1: "직접 파싱", 2: "마크다운 추출", 3: "중괄호 탐색"}
    print(f"{name}: {stage}단계({stage_names[stage]})에서 성공 -> {result}")

In [ ]:
# 검증 셀
_, stage1 = parse_json_response_with_stage(text1)
_, stage2 = parse_json_response_with_stage(text2)
_, stage3 = parse_json_response_with_stage(text3)

assert stage1 == 1, f"text1은 1단계에서 파싱되어야 합니다. 현재: {stage1}단계"
assert stage2 == 2, f"text2는 2단계에서 파싱되어야 합니다. 현재: {stage2}단계"
assert stage3 == 3, f"text3은 3단계에서 파싱되어야 합니다. 현재: {stage3}단계"
print("실습 3 완료!")

## 정리

이번 실습에서 학습한 내용:

1. **LLM 응답의 다양성**: 동일한 요청에도 JSON, 마크다운 코드블록, 자유 텍스트 등 다양한 형태로 응답
2. **3단계 폴백 전략**:
   - 1단계: `json.loads()` 직접 파싱 (가장 빠름)
   - 2단계: 마크다운 코드블록 regex 추출
   - 3단계: `{...}` 중괄호 탐색 (최후 수단)
3. **ValueError**: 모든 단계 실패 시 예외 발생 → 상위 레이어에서 처리
4. **with_structured_output() 우선**: 실제 LLM에서는 폴백보다 `with_structured_output()` 권장

**다음 모듈**: `module_07_llm_optimization/01_cost_calculator.ipynb` - LLM 비용 계산